# Predictability ceiling

Before any model comparison, this notebook quantifies what an oracle could do with this table, in four parts:

1. how much variance simple structure (route, calendar, weather) explains,
2. how much information the delay history itself carries,
3. where the error actually lives, and
4. how much wider the uncertainty is on bad-weather days.

The short version: most day-to-day variation is irreducible at this granularity, errors concentrate on severe-weather days, and uncertainty bands have to widen with weather to stay calibrated.

In [1]:
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.config import TABULAR_FEATURES, TRAIN_END
from src.features.registry import FEATURE_GROUPS

# lightgbm's sklearn wrapper warns about feature names when mixing frames
# and arrays; cosmetic here since column order is fixed by TABULAR_FEATURES
warnings.filterwarnings("ignore", message="X does not have valid feature names")

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
print(f"{len(df):,} rows, {df['route'].nunique()} routes, "
      f"{df['date'].min().date()} to {df['date'].max().date()}, "
      f"{len(TABULAR_FEATURES)} model features")

118,650 rows, 50 routes, 2019-01-01 to 2025-06-30, 80 model features


## Variance decomposition

Group-mean R-squared for one factor at a time: how much of the total variance in `avg_arr_delay` disappears if every row is predicted by its group average. This is descriptive (computed on all rows), meant to size the structure, not to validate a model.

In [2]:
target = df["avg_arr_delay"]
total_ss = ((target - target.mean()) ** 2).sum()


def group_r2(keys):
    group_mean = df.groupby(keys)["avg_arr_delay"].transform("mean")
    return 1 - ((target - group_mean) ** 2).sum() / total_ss


factors = {
    "route": ["route"],
    "day of week": ["day_of_week"],
    "month": ["month"],
    "weather severity (worst airport)": ["weather_severity_max"],
    "route + day of week": ["route", "day_of_week"],
    "route + month + weather severity": ["route", "month", "weather_severity_max"],
}
for name, keys in factors.items():
    print(f"{name:>34}: R2 = {group_r2(keys):.3f}")

                             route: R2 = 0.030
                       day of week: R2 = 0.008
                             month: R2 = 0.020
  weather severity (worst airport): R2 = 0.068
               route + day of week: R2 = 0.043
  route + month + weather severity: R2 = 0.173


Route identity, calendar, and daily weather each explain only a few percent of the variance, and even jointly they leave most of it untouched. The bulk of day-to-day delay variation is driven by things a daily route-level table cannot see: individual aircraft rotations, ATC flow programs, crew availability.

## Information content of the delay history

A linear regression on the nine delay-lag features bounds the linear information content of the target's own past. Tree models can squeeze out somewhat more via interactions, but not an order of magnitude more.

In [3]:
from sklearn.linear_model import LinearRegression

lag_cols = FEATURE_GROUPS["delay_lags"]
mask = df[lag_cols + ["avg_arr_delay"]].notna().all(axis=1)
X, y = df.loc[mask, lag_cols], df.loc[mask, "avg_arr_delay"]
r2 = LinearRegression().fit(X, y).score(X, y)
print(f"in-sample linear R2 from delay lags alone: {r2:.3f}")

in-sample linear R2 from delay lags alone: 0.239


## Where the error lives

A LightGBM fit on the training window with the tuned parameters makes the point concrete: severe-weather days (rain or worse at either endpoint, the same threshold the pipeline uses for its adversity flag) are a minority of rows but hold the majority of squared error.

In [4]:
params = json.loads(Path("../configs/best_params_lightgbm.json").read_text())
params.update({"max_depth": -1, "random_state": 42, "n_jobs": -1, "verbose": -1})

train = df[df["date"] < TRAIN_END].dropna(subset=TABULAR_FEATURES + ["avg_arr_delay"])
test = df[df["date"] >= TRAIN_END].dropna(subset=TABULAR_FEATURES + ["avg_arr_delay"])

model = lgb.LGBMRegressor(**params)
model.fit(train[TABULAR_FEATURES].values, train["avg_arr_delay"].values)
resid = test["avg_arr_delay"].values - model.predict(test[TABULAR_FEATURES].values)

severe = (test["weather_severity_max"] >= 3).values
sq = resid ** 2
print(f"severe-weather share of test rows:        {severe.mean():.1%}")
print(f"severe-weather share of squared error:    {sq[severe].sum() / sq.sum():.1%}")
print(f"test MAE overall: {np.abs(resid).mean():.2f} min, "
      f"severe days: {np.abs(resid[severe]).mean():.2f} min, "
      f"clear days: {np.abs(resid[~severe]).mean():.2f} min")

severe-weather share of test rows:        31.8%
severe-weather share of squared error:    62.9%
test MAE overall: 11.26 min, severe days: 16.00 min, clear days: 9.04 min


## Heteroskedasticity

The residual spread on severe days versus clear days sizes how much a fixed-width interval would lie.

In [5]:
std_severe = resid[severe].std()
std_clear = resid[~severe].std()
print(f"residual std, severe days: {std_severe:.1f} min")
print(f"residual std, clear days:  {std_clear:.1f} min")
print(f"ratio: {std_severe / std_clear:.2f}x")

residual std, severe days: 26.1 min
residual std, clear days:  13.7 min
ratio: 1.91x


## What this means

- The MAE differences between well-tuned models are small next to the noise floor above; chasing decimals on the point forecast is not where the value is.
- Error and uncertainty concentrate on severe-weather days, which is exactly where the aviation weather features (visibility, CAPE, gusts, freezing rain, low cloud) are aimed, and why the ablation reports severe-day metrics separately.
- A single point forecast understates risk precisely when it matters most. The quantile intervals exist because the noise is roughly twice as wide on bad days, and any interval calibration has to respect that.